# 02 — Training runs

**Main experiment:** 5 architectures × 3 split conditions × 3 seeds = 45 runs, as **one notebook copy per architecture**
(`RUN = "main"`, set `ARCH`; 9 runs each).
**Extra split realisations:** `RUN = "extra_splits"` trains all 5 architectures on `image_random_s1/s2` and
`grouped_s1/s2` with seed 0 (20 runs, one commit), to show the leakage gap does not depend on one split.

Each run saves val/test logits to `preds.csv` (final predictions in FP32); all metrics are computed later from those files.

**Setup:** dataset `paultimothymooney/chest-xray-pneumonia`, Internet **on**, accelerator **GPU T4 ×1** (or P100).

**How to run**
1. Set `RUN` (and `ARCH` for main) in the settings cell.
2. Interactive session: run everything with `SMOKE = True` (~5 min) to check that it works.
3. Set `SMOKE = False`, then **Save Version → Save & Run All (Commit)**. Runs in the background; outputs are kept.
   No new run starts after `BUDGET_H` hours, so the commit finishes and saves before the 12 h limit.
4. If runs are left over, attach the commit's output as an input to the next commit: finished runs are copied in and skipped.

In [ ]:
!pip install -q --no-deps --force-reinstall --no-cache-dir git+https://github.com/meap11/ppneu.git
try:
    import timm
except ImportError:
    !pip install -q timm

In [ ]:
import time
T0 = time.time()

SMOKE = True
# RUN = "main":         one architecture per notebook copy, 3 splits x 3 seeds (9 runs)
# RUN = "extra_splits": all 5 architectures on the 4 extra split realisations, seed 0 (20 runs, ~1.5 h)
RUN  = "main"
ARCH = "resnet50"   # main only: resnet50 | densenet121 | inception_v3 | mobilenetv3 | efficientnet_b0

ALL_ARCHS = ["resnet50", "densenet121", "inception_v3", "mobilenetv3", "efficientnet_b0"]
if RUN == "main":
    SPLITS, ARCHS, SEEDS = ["official", "image_random", "grouped"], [ARCH], [0, 1, 2]
elif RUN == "extra_splits":
    SPLITS = ["image_random_s1", "grouped_s1", "image_random_s2", "grouped_s2"]
    ARCHS, SEEDS = ALL_ARCHS, [0]
else:
    raise ValueError(RUN)
BUDGET_H = 11.0   # no new run starts if it might not finish by then (commit limit is 12 h)

In [ ]:
import os, shutil, json
from pathlib import Path
import pandas as pd, torch, timm
from ppneu import audit as A, data as D, train as T

print("torch", torch.__version__, "| timm", timm.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Turn on a GPU accelerator"

root, _ = A.find_root("/kaggle/input")
manifests = {s: D.load_manifest(s) for s in SPLITS}
# the official manifest lists all 5,856 images, so it builds the full cache
cache = D.build_cache(D.load_manifest("official"), root, "/kaggle/working/cache/cache256.npy",
                      workers=os.cpu_count())
print("cache", cache.shape)

In [ ]:
# Resume: copy finished runs from any attached previous output of this notebook.
# Kaggle mounts inputs at varying depths (/kaggle/input/<name>/ or
# /kaggle/input/notebooks/<user>/<name>/), so look a few levels down.
OUT = "/kaggle/working/smoke" if SMOKE else "/kaggle/working/runs"
if not SMOKE:
    prevs = sorted({p for pat in ("*/runs", "*/*/runs", "*/*/*/runs", "*/*/*/*/runs")
                    for p in Path("/kaggle/input").glob(pat) if p.is_dir()})
    print("previous outputs:", [str(p) for p in prevs] or "none")
    n = 0
    for prev in prevs:
        for preds in prev.glob("*/*/seed*/preds.csv"):
            dst = Path(OUT) / preds.parent.relative_to(prev)
            if not (dst / "preds.csv").exists():
                shutil.copytree(preds.parent, dst, dirs_exist_ok=True)
                n += 1
    print("resumed", n, "finished runs")

In [ ]:
if SMOKE:
    plan = [T.RunConfig(split=SPLITS[-1], arch=a, seed=0, epochs=2, limit=256,
                        workers=2, out_root=OUT) for a in ARCHS]
else:
    # seeds, then splits outermost, so a partial commit still covers every split
    plan = [T.RunConfig(split=s, arch=a, seed=k, workers=4, out_root=OUT)
            for k in SEEDS for s in SPLITS for a in ARCHS]
print(len(plan), "runs")

longest = 0.0
for i, cfg in enumerate(plan, 1):
    left = BUDGET_H * 3600 - (time.time() - T0)
    if left < 1.2 * longest:
        print(f"\nSTOP: {left / 3600:.1f} h left, longest run took {longest / 3600:.1f} h. "
              f"Attach this output to the next commit to resume.")
        break
    print(f"\n=== {i}/{len(plan)}: {cfg.split} / {cfg.arch} / seed {cfg.seed} ===")
    t = time.time()
    meta = T.train_one(cfg, manifests[cfg.split], cache, force=SMOKE)
    longest = max(longest, time.time() - t)
    print(f"best epoch {meta['best_epoch']} | val AUROC {meta['val_auc']:.4f} | "
          f"{meta['minutes']:.1f} min")

In [ ]:
summary = T.collect(OUT)
summary.to_csv(Path(OUT) / "summary.csv", index=False)
display(summary)
if not SMOKE and len(summary):
    display(summary.pivot_table(index="arch", columns="split", values="test_auc",
                                aggfunc=["mean", "std"]).round(4))

In [ ]:
# Free space: the ~0.4 GB image cache is not needed in the saved output
shutil.rmtree("/kaggle/working/cache", ignore_errors=True)
!du -sh /kaggle/working/*